In [1]:
import glob
import numpy as np
import pandas as pd
import seaborn as sns
import mne
from matplotlib import pyplot as plt
%matplotlib qt 

OFFSET = 0.015625

In [2]:
""" 
Set eeg stuff up
"""

# Load the 10-20 montage
montage = mne.channels.make_standard_montage('biosemi64')
# Get the list of channel names
electrode_names = montage.ch_names
sfreq = 512
info = mne.create_info(ch_names = electrode_names,sfreq = sfreq, ch_types='eeg')

max_time = .6 # change this depending on the ERP you want to look at
min_time = -.1

condition_columns = ['feature','attended_feature','unattended_feature','visual_field']
non_ch_order = ['subject', 'time', 'feature','attended_feature','unattended_feature','visual_field']
col_order = non_ch_order + electrode_names

In [3]:
def create_evokeds(df, condition_columns, channel_columns, info):
    """
    Function create evokeds per condition
    """
    evokeds = {}
    # Group by all condition columns and time
    grouped = df.groupby(condition_columns + ['time'])[channel_columns].mean()
    
    # Get unique condition combinations
    condition_combinations = df[condition_columns].drop_duplicates().values
    
    for combo in condition_combinations:
        # Create condition name
        condition_name = ' / '.join([f"{col}: {val}" for col, val in zip(condition_columns, combo)])
        # Create tuple for indexing
        index_tuple = tuple(combo)
        # Extract data for this condition
        condition_data = grouped.loc[index_tuple].sort_index()
        data = condition_data.values.T
        # Create evoked object
        ev = mne.EvokedArray(data, info, tmin=condition_data.index[0])
        ev.set_montage(montage)
        ev.apply_baseline()
        ev.info['bads'] = ['T8']
        ev.interpolate_bads(reset_bads=True)
        evokeds[condition_name] = ev

    return evokeds

In [4]:
""" 
Main data
"""
sub_path = "C:/Users/mvmigem/Documents/data/project_2/overlap_corrected/main/"
dir_list = glob.glob(sub_path+'/*.csv')
evokeds_sub = []

for p in dir_list:
    df_sub = pd.read_csv(p)
    df_sub = df_sub[df_sub['sequence'] == 2]
    # Make the updwon col
    df_sub['visual_field'] = np.where(df_sub['position'].isin([1,2]),'up','down')
    # Drop non EEG channels
    df_sub = df_sub[~df_sub['channel'].isin(['eye_above','eye_below','eye_left','eye_right','Status'])]
    df_sub['time'] = df_sub['time'] - OFFSET
    df_sub = df_sub[df_sub['time'] < max_time]
    df_sub = df_sub[df_sub['time'] > min_time]
    df_wide_sub = df_sub.pivot_table(
        index=non_ch_order,
        columns='channel',
        values='yhat',
        aggfunc='first'  # use 'first' if no duplicates, or 'mean'/'sum' if needed
    ).reset_index()
    df_wide_sub = df_wide_sub[col_order]
    # create evoked for every condition
    evokeds = create_evokeds(df_wide_sub,condition_columns=condition_columns,channel_columns=electrode_names,info=info)
    evokeds_sub.append(evokeds)
    


Applying baseline correction (mode: mean)
Setting channel interpolation method to {'eeg': 'spline'}.
Interpolating bad channels.
    Automatic origin fit: head of radius 95.0 mm
Computing interpolation matrix from 63 sensor positions
Interpolating 1 sensors
Applying baseline correction (mode: mean)
Setting channel interpolation method to {'eeg': 'spline'}.
Interpolating bad channels.
    Automatic origin fit: head of radius 95.0 mm
Computing interpolation matrix from 63 sensor positions
Interpolating 1 sensors
Applying baseline correction (mode: mean)
Setting channel interpolation method to {'eeg': 'spline'}.
Interpolating bad channels.
    Automatic origin fit: head of radius 95.0 mm
Computing interpolation matrix from 63 sensor positions
Interpolating 1 sensors
Applying baseline correction (mode: mean)
Setting channel interpolation method to {'eeg': 'spline'}.
Interpolating bad channels.
    Automatic origin fit: head of radius 95.0 mm
Computing interpolation matrix from 63 sensor po

In [5]:
def compute_grand_averages(list_of_evoked_dicts):
    """
    Compute grand averages for each condition across subjects
    """
    # Get all condition names (should be the same for all subjects)
    condition_names = list(list_of_evoked_dicts[0].keys())
    
    grand_averages = {}
    
    for condition in condition_names:
        # Extract all evoked objects for this condition across subjects
        evokeds_for_condition = [subj_dict[condition] for subj_dict in list_of_evoked_dicts]
        
        # Compute grand average
        grand_avg = mne.grand_average(evokeds_for_condition)
        grand_averages[condition] = grand_avg
    
    return grand_averages

# Usage
grand_averages = compute_grand_averages(evokeds_sub)

Identifying common channels ...
Identifying common channels ...
Identifying common channels ...
Identifying common channels ...
Identifying common channels ...
Identifying common channels ...
Identifying common channels ...
Identifying common channels ...
Identifying common channels ...
Identifying common channels ...
Identifying common channels ...
Identifying common channels ...
Identifying common channels ...
Identifying common channels ...
Identifying common channels ...
Identifying common channels ...


In [6]:
a1 = grand_averages['feature: rotation / attended_feature: regular / unattended_feature: regular / visual_field: down']
a2 = grand_averages['feature: rotation / attended_feature: odd / unattended_feature: regular / visual_field: down']
b1 = grand_averages['feature: rotation / attended_feature: regular / unattended_feature: odd / visual_field: down']
b2 = grand_averages['feature: rotation / attended_feature: odd / unattended_feature: odd / visual_field: down']
c1 = grand_averages['feature: rotation / attended_feature: regular / unattended_feature: regular / visual_field: up']
c2 = grand_averages['feature: rotation / attended_feature: odd / unattended_feature: regular / visual_field: up']
d1 = grand_averages['feature: rotation / attended_feature: regular / unattended_feature: odd / visual_field: up']
d2 = grand_averages['feature: rotation / attended_feature: odd / unattended_feature: odd / visual_field: up']

a1a = grand_averages['feature: angle / attended_feature: regular / unattended_feature: regular / visual_field: down']
a2a = grand_averages['feature: angle / attended_feature: odd / unattended_feature: regular / visual_field: down']
b1a = grand_averages['feature: angle / attended_feature: regular / unattended_feature: odd / visual_field: down']
b2a = grand_averages['feature: angle / attended_feature: odd / unattended_feature: odd / visual_field: down']
c1a = grand_averages['feature: angle / attended_feature: regular / unattended_feature: regular / visual_field: up']
c2a = grand_averages['feature: angle / attended_feature: odd / unattended_feature: regular / visual_field: up']
d1a = grand_averages['feature: angle / attended_feature: regular / unattended_feature: odd / visual_field: up']
d2a = grand_averages['feature: angle / attended_feature: odd / unattended_feature: odd / visual_field: up']

a = mne.combine_evoked([a1,a2],weights='nave')
b = mne.combine_evoked([b1,b2],weights='nave')
c = mne.combine_evoked([c1,c2],weights='nave')
d = mne.combine_evoked([d1,d2],weights='nave')

# diff_ab = mne.combine_evoked([a,b],weights=[1,-1])
# diff_cd = mne.combine_evoked([c,d],weights=[1,-1])
ac = mne.combine_evoked([a,c],weights='nave')
bd = mne.combine_evoked([b,d],weights='nave')

diff = mne.combine_evoked([ac,bd],weights=[1,-1])

a_all = mne.combine_evoked([a1,a2,a1a,a2a],weights='nave')
b_all = mne.combine_evoked([b1,b2,b1a,b2a],weights='nave')
c_all = mne.combine_evoked([c1,c2,c1a,c2a],weights='nave')
d_all = mne.combine_evoked([d1,d2,d1a,d2a],weights='nave')

ac_all = mne.combine_evoked([a_all,c_all],weights='nave')
bd_all = mne.combine_evoked([b_all,d_all],weights='nave')

diff_all = mne.combine_evoked([ac_all,bd_all],weights=[1,-1])

# down = mne.combine_evoked([a,b],weights='nave')
# up = mne.combine_evoked([c,d],weights='nave')

# regular_down = mne.combine_evoked([a1,b1],weights='nave')
# odd_down = mne.combine_evoked([a2,b2],weights='nave')
# regular_up = mne.combine_evoked([c1,d1],weights='nave')
# odd_up = mne.combine_evoked([c2,d2],weights='nave')

# regular = mne.combine_evoked([regular_down,regular_up],weights='nave')
# odd = mne.combine_evoked([odd_down,odd_up],weights='nave')

In [10]:
rot_areg = mne.combine_evoked([a1,b1,c1,d1],weights='nave')
rot_aodd = mne.combine_evoked([a2,b2,c2,d2],weights='nave')
ori_areg = mne.combine_evoked([a1a,b1a,c1a,d1a],weights='nave')
ori_aodd = mne.combine_evoked([a2a,b2a,c2a,d2a],weights='nave')

diff_rot_a = mne.combine_evoked([rot_areg,rot_aodd],weights=[1,-1])
diff_ori_a = mne.combine_evoked([ori_areg,ori_aodd],weights=[1,-1])

a_reg = mne.combine_evoked([rot_areg,ori_areg],weights='nave')
a_odd = mne.combine_evoked([rot_aodd,ori_aodd],weights='nave')

diff_a = mne.combine_evoked([a_reg,a_odd],weights=[1,-1])

In [11]:
diff.info['bads'] = ['T7','AF7','Fp1']
diff.interpolate_bads(reset_bads=True)

diff_all.info['bads'] = ['T7','FT7']
diff_all.interpolate_bads(reset_bads=True)

diff_ori_a.info['bads'] = ['FT7']
diff_ori_a.interpolate_bads(reset_bads=True)

diff_a.info['bads'] = ['TP7','T7','FT7','AF7','Fp1']
diff_a.interpolate_bads(reset_bads=True)

Setting channel interpolation method to {'eeg': 'spline'}.
Interpolating bad channels.
    Automatic origin fit: head of radius 95.0 mm
Computing interpolation matrix from 61 sensor positions
Interpolating 3 sensors
Setting channel interpolation method to {'eeg': 'spline'}.
Interpolating bad channels.
    Automatic origin fit: head of radius 95.0 mm
Computing interpolation matrix from 62 sensor positions
Interpolating 2 sensors
Setting channel interpolation method to {'eeg': 'spline'}.
Interpolating bad channels.
    Automatic origin fit: head of radius 95.0 mm
Computing interpolation matrix from 63 sensor positions
Interpolating 1 sensors
Setting channel interpolation method to {'eeg': 'spline'}.
Interpolating bad channels.
    Automatic origin fit: head of radius 95.0 mm
Computing interpolation matrix from 59 sensor positions
Interpolating 5 sensors


<Evoked | '(0.500 × (0.250 × Grand average (n = 32) + 0.250 × Grand average (n = 32) + 0.250 × Grand average (n = 32) + 0.250 × Grand average (n = 32)) + 0.500 × (0.250 × Grand average (n = 32) + 0.250 × Grand average (n = 32) + 0.250 × Grand average (n = 32) + 0.250 × Grand average (n = 32))) - (0.500 × (0.250 × Grand average (n = 32) + 0.250 × Grand average (n = 32) + 0.250 × Grand average (n = 32) + 0.250 × Grand average (n = 32)) + 0.500 × (0.250 × Grand average (n = 32) + 0.250 × Grand average (n = 32) + 0.250 × Grand average (n = 32) + 0.250 × Grand average (n = 32)))' (average, N=128.0), -0.099609 – 0.59961 s, baseline -0.0996094 – 0 s, 64 ch, ~255 KiB>

In [13]:
times = np.arange(0.34, 0.35, 0.02)
vlim = (1.5, -1.5)
# vlim = (None, None)

# fig = a.plot_topomap(ch_type="eeg", times=times, colorbar=True,vlim = vlim)
# fig.suptitle("Rotation - unattended feature : Regular - Lower visual field")
# fig = b.plot_topomap(ch_type="eeg", times=times, colorbar=True,vlim = vlim)
# fig.suptitle("Rotation - unattended feature : Odd - Lower visual field")
# fig = b.plot_topomap(ch_type="eeg", times=times, colorbar=True,vlim = vlim)
# fig.suptitle("Rotation - unattended feature : Regular - Upper visual field")
# fig = d.plot_topomap(ch_type="eeg", times=times, colorbar=True,vlim = vlim)
# fig.suptitle("Rotation - unattended feature : Odd - Upper visual field")
# fig = diff_ab.plot_topomap(ch_type="eeg", times=times, colorbar=True,vlim = vlim)
# fig.suptitle("Diff unattended feature Regular - odd  / Lower visual field")
# fig = diff_cd.plot_topomap(ch_type="eeg", times=times, colorbar=True,vlim = vlim)
# fig.suptitle("Diff unattended feature Regular - odd  / Upper visual field")
# fig = up.plot_topomap(ch_type="eeg", times=times, colorbar=True,vlim = vlim)
# fig = down.plot_topomap(ch_type="eeg", times=times, colorbar=True,vlim = vlim)
# fig = regular.plot_topomap(ch_type="eeg", times=times, colorbar=True,vlim = vlim)
# fig = odd.plot_topomap(ch_type="eeg", times=times, colorbar=True,vlim = vlim)

# fig = ac.plot_topomap(ch_type="eeg", times=times, colorbar=True,vlim = vlim)
# fig = bd.plot_topomap(ch_type="eeg", times=times, colorbar=True,vlim = vlim)

# fig = diff_all.plot_topomap(ch_type="eeg", times=times, colorbar=True,vlim = vlim)
# fig = diff.plot_topomap(ch_type="eeg", times=times, colorbar=True,vlim = vlim)

# fig = diff_ori_a.plot_topomap(ch_type="eeg", times=times, colorbar=True,vlim = vlim)
# fig = diff_rot_a.plot_topomap(ch_type="eeg", times=times, colorbar=True,vlim = vlim)

fig = diff_a.plot_topomap(ch_type="eeg", times=times, colorbar=True,vlim = vlim)


In [14]:
# Convert dictionary values to list
all_evokeds = list(grand_averages.values())
    
# Average all conditions together
overall_average = mne.grand_average(all_evokeds)
    


Identifying common channels ...


In [16]:
times = np.arange(0.3, 0.45, 0.02)
vlim = (None, None)

fig = overall_average.plot_topomap(ch_type="eeg", times=times, colorbar=True,)